# CoWork Enterprise Demo — Build & Harden the Agent
## Notebook 03 — Build & Harden the Agent (Phase 3)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ Creates a **Cortex Agent** from a YAML **specification** (agent-as-code)
2. 🛠️ Wires it to the semantic view (structured) + Cortex Search (unstructured) with precise **tool descriptions**
3. 🛠️ Grants the final access layer (**`USAGE ON AGENT`**) to the consumer role
4. 🛠️ Tests the agent with `DATA_AGENT_RUN` and confirms account **guardrails** + the **audit trail**

---

### Why Tool Descriptions Are the #1 Lever:

🔹 A **Cortex Agent** orchestrates across tools to answer questions: it **plans** (interprets intent), **selects a tool** (Analyst for SQL, Search for RAG), **reflects**, and **responds** with a grounded answer.

The agent chooses tools by reading their **descriptions** — so clear, specific descriptions (what the tool does, what data, *when to use / when not to use*) matter more than any other single setting. Keep agents **narrow** (one clear purpose); narrow agents outperform catch-all agents.

**Agent anatomy** (defined in the `FROM SPECIFICATION` YAML):

| Component | What it does |
|---|---|
| `models` | The LLM that reasons (`auto` = best available) |
| `orchestration.budget` | Per-request time/token cap — a cost + latency guardrail |
| `instructions.response` | Tone/format of answers |
| `instructions.orchestration` | **When to use which tool** |
| `tools` + `tool_resources` | The tools and the objects they point at |

> **Documentation:** [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

---

### Estimated Time: **20–25 minutes**

### Prerequisites:
- Notebook 02 complete (consumer role + masking in place)

## Step 1: Set Context

In [ ]:
%%sql -r set_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA AGENTS;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


## Step 2: Create the Agent (from specification)

🛠️ `CREATE OR REPLACE AGENT ... FROM SPECIFICATION` defines the agent as code. Note two things in the spec:

🔹 **`tool_resources.execution_environment`** — agents are **serverless**; they do *not* inherit your session's `USE WAREHOUSE`. You must name the warehouse the generated SQL runs on, right in the spec, so the agent works when called from CoWork or the REST API.

🔹 **Tool descriptions** — the Analyst and Search tools each carry a description telling the agent exactly when to use them. This is the primary driver of correct routing.

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT
  COMMENT = 'CoWork Enterprise Demo agent - clients, portfolios, trades, and research'
  PROFILE = '{"display_name": "Enterprise Demo Analyst", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Enterprise Demo Analyst. You help portfolio managers, relationship managers,
    and compliance officers understand client portfolios, trading activity, and market research.

    Guidelines:
    - Be concise and data-driven. Lead with numbers when available.
    - When showing financial data, format large numbers (e.g., $2.5B not $2500000000).
    - If a question spans both structured data (portfolios, trades) and unstructured data
      (research notes), use BOTH tools to provide a complete answer.
    - Always cite which data source your answer came from.
    - For compliance questions, flag any potential issues clearly.
  orchestration: |
    - For questions about client AUM, portfolio values, positions, or trades: use the Analyst tool.
    - For questions about market outlook, research opinions, or compliance reports: use the Search tool.
    - For questions that need both, use both tools.
  sample_questions:
    - question: "Who are our top 5 clients by AUM?"
    - question: "What is our total technology sector exposure?"
    - question: "Are there any compliance concerns I should know about?"
    - question: "Show me recent trades over $1M and the rationale behind them."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "nexus_analytics"
      description: "Query structured financial data including client accounts, portfolio positions, trade history, and business metrics. Use this for any question about numbers, rankings, aggregations, or trends."
  - tool_spec:
      type: "cortex_search"
      name: "nexus_research"
      description: "Search analyst research notes, market commentary, risk assessments, and compliance reports. Use this for qualitative insights, opinions, recommendations, and compliance flags."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generate visualizations from query results when the user asks for charts or visual breakdowns."

tool_resources:
  nexus_analytics:
    semantic_view: "COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV"
    execution_environment:
      type: warehouse
      warehouse: "COWORK_ENTERPRISE_DEMO_WH"
  nexus_research:
    name: "COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_SEARCH"
    max_results: "5"
$$;
SELECT 'Agent DEMO_AGENT created' AS STATUS;


## Step 3: Grant the Final Access Layer (Tier 2)

🛠️ The agent now exists, so we can grant **`USAGE ON AGENT`** to the consumer role — completing the four-layer model from Notebook 02.

In [ ]:
%%sql -r grant_agent_usage
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
SELECT 'Tier 2 agent USAGE granted' AS STATUS;


In [ ]:
%%sql -r describe_agent
DESCRIBE AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT;


### 🔹 Agent Identity: How the Agent Is Identified When It Acts

A Cortex Agent has **no service account of its own** — it acts with the identity of whoever calls it. That single fact defines its whole security posture:

| Facet | How it works here |
|---|---|
| **Runtime identity** | **Caller's rights** — every tool call and generated SQL runs as the invoking user's role, inheriting their grants, masking, and row-access (Notebook 02). |
| **Object identity** | The agent is a **securable object** with an owner; access is `USAGE ON AGENT` (the Tier-2 grant above). |
| **Display identity** | The `PROFILE` (name, avatar, colour) users see in CoWork (branding is finalized in Notebook 05). |
| **Outbound identity** | For external actions via **MCP**, calls use **per-user OAuth** — the agent acts as the user in the target system too (see the MCP note below; illustrative). |
| **Attribution** | Every interaction is logged to a `USER_NAME` in Account Usage (Step 6) — fully traceable, since there's no shared service principal to hide behind. |

📌 **Implication:** you govern the *agent* by governing *users and RBAC* — there is no separate agent credential to manage or rotate. This is the Snowflake Security Framework's *Agent Identity* pillar in practice.

### 🔮 Restricting the Agent *Below* the Caller (Preview)

⚠️ **Preview / reference — do not run unless these capabilities are enabled in your account.**

RBAC answers *"what can the user do?"* An emerging pattern adds *"even if the user can, should the **agent** be allowed to do it automatically?"* — a tighter boundary for AI-driven execution than for a human at a worksheet.

- **Agent Policy** — an account-level control that can restrict an agent session **below** the caller's role (e.g. read-only on one database), independent of what the user's role otherwise allows.
- **`SYS_CONTEXT`** — exposes the acting agent's identity inside a session, so masking / row-access policies can branch on *whether an agent is calling*, not just on which role.

📌 Until these are GA in your account, achieve the same intent with the layered controls already in this guide: a **narrow consumer role**, **tool scoping** in the spec, **guardrails**, and **budgets / quota** (Notebook 04).

## Step 4: Test the Agent

🛠️ `DATA_AGENT_RUN` calls the agent programmatically (the same engine CoWork uses). It should **ground** its answer in the semantic view and cite its source.

📌 Faster iteration: **AI & ML → Agents → DEMO_AGENT → Test** gives a chat UI with no JSON.

In [ ]:
%%sql -r test_agent
-- Test 1: Client ranking (uses Cortex Analyst → semantic view → SQL)
SELECT
  TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Who are our top 5 clients by AUM and what regions are they in?" }
            ]
          }
        ],
        "stream": false
      }$$
    )
  ) AS resp;


### 🔹 A Note on `DATA_AGENT_RUN` Output

`SNOWFLAKE.CORTEX.DATA_AGENT_RUN` returns the agent's response as a **string**. Three patterns show up in practice:

| Pattern | Behaviour | Best for |
|---|---|---|
| `DATA_AGENT_RUN(...)` | Raw string | Quick testing — errors are visible as text |
| `PARSE_JSON(DATA_AGENT_RUN(...))` | Parses to JSON; errors on invalid JSON | Debugging — forces failures to surface |
| `TRY_PARSE_JSON(DATA_AGENT_RUN(...))` | Parses to JSON; NULL on bad JSON (never errors) | Structured output — extract fields |

🔹 The parsed response is an **object** with a `content` array (text blocks + tool blocks). To read just the answer, flatten `content` and keep the `text` rows:

```sql
SELECT f.value:text::string AS answer
FROM TABLE(FLATTEN(input => TRY_PARSE_JSON(DATA_AGENT_RUN('...', '{...}')):content)) f
WHERE f.value:type::string = 'text';
```

## Step 5: Guardrails (account-global — read-only here)

🔹 **Cortex AI Guardrails** (prompt-injection / jailbreak protection) are set at the **account** level and apply across Cortex Agents, CoWork, and CoCo. On this shared account they are **already enabled**, so we only read the setting — we never overwrite another team's config.

```sql
-- Reference: enable guardrails (ACCOUNTADMIN, Enterprise Edition)
ALTER ACCOUNT SET AI_SETTINGS = '{"guardrails":{"prompt_injection":{"enabled":true},"jailbreak":{"enabled":true}}}';
```

**Do you actually need to enable them?** Modern foundation models (Claude 4, GPT-4.1) have strong built-in safety, so native behaviour is often enough for **internal, read-only, narrowly-scoped** agents. Turn guardrails **on** when any of these apply:

- **External-facing** agents (customers / partners have access, not just employees)
- **Adversarial users** are plausible, or the cost of a single bypass is high (financial / legal / reputational)
- **Compliance** mandates defense-in-depth

🔹 Guardrails are **additive** (charged per token scanned), not mandatory — they layer on top of the RBAC and tool-scoping you already have.

In [ ]:
%%sql -r show_guardrails
SHOW PARAMETERS LIKE 'AI_SETTINGS' IN ACCOUNT;


## MCP governance (illustrative - external OAuth required)
If the agent needs to take actions in external systems (e.g. create a Jira ticket), connect via a governed MCP server. This needs an external OAuth app, so it is shown for reference and **not run** here.

```sql
CREATE API INTEGRATION demo_mcp_integration
  API_PROVIDER = external_mcp
  API_ALLOWED_PREFIXES = ('https://mcp.example.com')
  API_USER_AUTHENTICATION = (TYPE = OAUTH_DYNAMIC_CLIENT,
    OAUTH_RESOURCE_URL = 'https://mcp.example.com/v1/mcp')
  ENABLED = TRUE;

CREATE EXTERNAL MCP SERVER COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_MCP
  WITH DISPLAY_NAME = 'Demo MCP'
  URL = 'https://mcp.example.com/v1/mcp'
  API_INTEGRATION = demo_mcp_integration;

GRANT USAGE ON EXTERNAL MCP SERVER COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_MCP TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;

-- Governance: USAGE controls who can connect; actions are per-user OAuth scoped;
-- every tool call is logged in Account Usage. Kill switch:
ALTER API INTEGRATION demo_mcp_integration SET ENABLED = FALSE;
```

**Available connectors** (Snowflake-hosted proxy endpoints — Snowflake handles the MCP protocol and OAuth token management):

| Connector | Auth | Setup |
|---|---|---|
| Atlassian (Jira/Confluence), Glean, Linear | Dynamic client registration | Easiest — no manual OAuth app |
| GitHub, Salesforce | Standard OAuth | Create an OAuth / connected app |
| Gmail, Google Drive/Calendar, Slack | Standard OAuth | Create the provider OAuth app (or one-click Browse Connectors on Snowhouse) |
| Custom | OAuth2 (manual) | Any MCP endpoint |

**MCP security model:**

| Concern | How Snowflake handles it |
|---|---|
| Who can connect? | `USAGE` on the MCP server object — RBAC controlled |
| What data leaves Snowflake? | Only the tool-call request + response — not raw table data |
| Credentials | OAuth tokens managed by Snowflake, per-user, auto-rotating |
| Audit | Every MCP tool call logged in Account Usage |
| Kill switch | `ALTER API INTEGRATION ... SET ENABLED = FALSE` revokes all tokens instantly |

> **Documentation:** [MCP Connectors — key considerations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-mcp-connectors)

### 🔹 The Trust Boundary — What Stays in Snowflake vs. Goes External

```
  +------------- SNOWFLAKE TRUST BOUNDARY -------------+
  |  question -> agent orchestration -> tool select   |
  |    Cortex Analyst (SQL)    Cortex Search (RAG)     |
  |    your data stays here    your data stays here    |
  |    cross-region inference: payload is transient,   |
  |    encrypted, and never persisted                  |
  +----------------------------+-----------------------+
                               | MCP tool call (external)
                               v
     External service (e.g. Jira): receives only the tool
     request + response via per-user OAuth — never raw tables
```

- **Data at rest never leaves your region** — tables, stages, and stored data stay put.
- **Inference payloads** (prompt + response) may route cross-region, but are transient, encrypted (TLS 1.2+, mTLS cross-region), and never stored. **No egress charge.**
- **MCP calls** send only the tool invocation and its result — not your underlying data.
- **Pricing:** cross-region is governed by `CORTEX_ENABLED_CROSS_REGION` — Global (`ANY_REGION`) bills at a lower per-AI-credit rate than Regional; we enabled it in Notebook 00.

> **Documentation:** [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

## Step 6: Audit Trail

📌 Every agent interaction is logged to Account Usage and attributable to an individual user (CoWork uses no service account). Usage data can take a short while to appear.

In [ ]:
%%sql -r audit_agent
SELECT START_TIME, USER_NAME, AGENT_NAME, TOKENS, TOKEN_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE AGENT_NAME = 'DEMO_AGENT'
ORDER BY START_TIME DESC
LIMIT 10;


### 📌 Quick Reference — How to Lock Down AI Access

| Goal | Command |
|---|---|
| Disable all AI for a role | `REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE <role>` |
| Restrict who can use the agent | grant `USAGE ON AGENT` only to approved roles |
| Scope the agent's SQL to specific tables | grant the role `SELECT` on just those tables (not schema-wide) |
| Disable a specific MCP connector | `ALTER API INTEGRATION <name> SET ENABLED = FALSE` |
| Turn off cross-region inference | `ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED'` |
| Enable prompt-injection / jailbreak guardrails | `ALTER ACCOUNT SET AI_SETTINGS = '{...guardrails...}'` |
| Cap per-user AI credits | `SNOWFLAKE.CORE.QUOTA` + `SET_PER_USER_LIMIT` + block enforcement (NB04) |
| Cap team / agent credits | `SNOWFLAKE.CORE.BUDGET` + `SET_SPENDING_LIMIT` (NB04) |

🔹 **Defense in depth:** RBAC + agent scoping + guardrails + PII masking + budgets / quota + audit trail — no single layer is the whole answer.

## ✅ Notebook 03 Complete

### What You Just Built:
- Agent `COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT` created from a versionable YAML spec, wired to the semantic view + search service
- `USAGE ON AGENT` granted to the consumer role (four-layer model complete)
- A verified `DATA_AGENT_RUN` test, guardrails posture, and audit trail

---

### Key Points:
- Tool descriptions are the #1 lever for agent accuracy — be specific about when to use each tool.
- Agents are serverless: the warehouse must be named in the spec, not inherited from the session.
- Guardrails and audit are account-native — every interaction is traceable to a user.

---

### Next:

Continue to **Notebook 04 — Control the Cost**: make AI spend visible, attribute it with tags, and cap it with budgets and a per-user quota.